
# C2ST, α-Precision и β-Recall


In [11]:
#synthcity310

In [ ]:
!pip install sdmetrics synthcity tqdm


In [2]:
import json
import math
from pathlib import Path
from typing import Dict, Tuple, List
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from tqdm.auto import tqdm
from synthcity.metrics import eval_statistical
from synthcity.plugins.core.dataloader import GenericDataLoader
from sdmetrics.single_table import LogisticDetection


[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.


In [12]:
!python -c "import torch; print(torch.__version__); print(torch.backends.mps.is_available())"

2.2.2
True


In [13]:
PROJECT_ROOT = Path(".").resolve()

DATA_DIR = PROJECT_ROOT / "data"

MODEL_DIRS = {
    "tabsyn": PROJECT_ROOT / "tabsyn",
    "tabdiff": PROJECT_ROOT / "tabdiff",
    "tabcsdi": PROJECT_ROOT / "tabcsdi",
    "catboost": PROJECT_ROOT / "catboost"
}

DATASETS = ["adult", "default", "shoppers", "magic", "beijing"]

OUTPUT_DIR = PROJECT_ROOT / "den_evaluation_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "c2st_alpha_beta_results.csv"

print("PROJECT_ROOT", PROJECT_ROOT)
print("DATA_DIR", DATA_DIR)
print("OUTPUT_CSV", OUTPUT_CSV)

PROJECT_ROOT /Users/romandegtarev/generative_tabels
DATA_DIR /Users/romandegtarev/generative_tabels/data
OUTPUT_CSV /Users/romandegtarev/generative_tabels/den_evaluation_results/c2st_alpha_beta_results.csv


In [14]:
def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(sparse_output=False)
    except TypeError:
        return OneHotEncoder(sparse=False)


def load_info(dataset: str) -> dict:
    info_path = DATA_DIR / dataset / "info.json"
    with open(info_path, "r", encoding="utf-8") as f:
        info = json.load(f)
    return info



def read_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df


def get_column_groups(info: dict) -> Tuple[List[int], List[int]]:
    """
    - если regression, target идет в num
    - иначе target идет в cat
    """
    num_col_idx = list(info["num_col_idx"])
    cat_col_idx = list(info["cat_col_idx"])
    target_col_idx = list(info["target_col_idx"])

    if info["task_type"] == "regression":
        num_col_idx += target_col_idx
    else:
        cat_col_idx += target_col_idx

    return num_col_idx, cat_col_idx


In [15]:
def prepare_alpha_beta_inputs(
    train_data: pd.DataFrame,
    real_data: pd.DataFrame,
    syn_data: pd.DataFrame,
    info: dict,
) -> Tuple[pd.DataFrame, pd.DataFrame]:

    train_data = train_data.copy()
    real_data = real_data.copy()
    syn_data = syn_data.copy()

    train_data.columns = range(len(train_data.columns))
    real_data.columns = range(len(real_data.columns))
    syn_data.columns = range(len(syn_data.columns))

    num_col_idx, cat_col_idx = get_column_groups(info)

    # numerical
    num_real_np = real_data[num_col_idx].to_numpy()
    num_syn_np = syn_data[num_col_idx].to_numpy()

    # categorical
    cat_train_np = train_data[cat_col_idx].to_numpy().astype(str)
    cat_real_np = real_data[cat_col_idx].to_numpy().astype(str)
    cat_syn_np = syn_data[cat_col_idx].to_numpy().astype(str)

    encoder = make_one_hot_encoder()
    encoder.fit(cat_train_np)   # <-- fit на train

    cat_real_oh = encoder.transform(cat_real_np)
    cat_syn_oh = encoder.transform(cat_syn_np)

    le_real_data = pd.DataFrame(
        np.concatenate((num_real_np, cat_real_oh), axis=1)
    ).astype(float)

    le_syn_data = pd.DataFrame(
        np.concatenate((num_syn_np, cat_syn_oh), axis=1)
    ).astype(float)

    return le_real_data, le_syn_data


def compute_alpha_beta(
    train_data: pd.DataFrame,
    real_data: pd.DataFrame,
    syn_data: pd.DataFrame,
    info: dict,
) -> Dict[str, float]:

    le_real_data, le_syn_data = prepare_alpha_beta_inputs(
        train_data, real_data, syn_data, info
    )

    X_syn_loader = GenericDataLoader(le_syn_data)
    X_real_loader = GenericDataLoader(le_real_data)

    quality_evaluator = eval_statistical.AlphaPrecision()
    qual_res = quality_evaluator.evaluate(X_real_loader, X_syn_loader)

    qual_res = {k: v for (k, v) in qual_res.items() if "naive" in k}

    return {
        "alpha_precision": float(qual_res["delta_precision_alpha_naive"]),
        "beta_recall": float(qual_res["delta_coverage_beta_naive"]),
    }

In [16]:
def reorder_for_c2st(
    real_data: pd.DataFrame,
    syn_data: pd.DataFrame,
    info: dict,
) -> Tuple[pd.DataFrame, pd.DataFrame, dict]:

    num_col_idx, cat_col_idx = get_column_groups(info)

    real_num_data = real_data[num_col_idx]
    real_cat_data = real_data[cat_col_idx]
    new_real_data = pd.concat([real_num_data, real_cat_data], axis=1)
    new_real_data.columns = range(len(new_real_data.columns))

    syn_num_data = syn_data[num_col_idx]
    syn_cat_data = syn_data[cat_col_idx]
    new_syn_data = pd.concat([syn_num_data, syn_cat_data], axis=1)
    new_syn_data.columns = range(len(new_syn_data.columns))

    metadata = json.loads(json.dumps(info["metadata"]))
    columns = metadata["columns"]
    metadata["columns"] = {}

    # переупорядочивает metadata под новый порядок колонок
    for i in range(len(new_real_data.columns)):
        if i < len(num_col_idx):
            metadata["columns"][i] = columns[str(num_col_idx[i])] if str(num_col_idx[i]) in columns else columns[num_col_idx[i]]
        else:
            src_idx = cat_col_idx[i - len(num_col_idx)]
            metadata["columns"][i] = columns[str(src_idx)] if str(src_idx) in columns else columns[src_idx]

    return new_real_data, new_syn_data, metadata


def compute_c2st(
    real_data: pd.DataFrame,
    syn_data: pd.DataFrame,
    info: dict,
) -> float:

    real_data = real_data.copy()
    syn_data = syn_data.copy()

    real_data.columns = range(len(real_data.columns))
    syn_data.columns = range(len(syn_data.columns))

    new_real_data, new_syn_data, metadata = reorder_for_c2st(real_data, syn_data, info)

    score = LogisticDetection.compute(
        real_data=new_real_data,
        synthetic_data=new_syn_data,
        metadata=metadata,
    )
    return float(score)


In [17]:

def load_train_real_and_prediction(dataset: str, model_name: str):
    dataset_dir = DATA_DIR / dataset

    train_path = dataset_dir / "train.csv"
    test_path = dataset_dir / "test.csv"
    info_path = dataset_dir / "info.json"

    pred_path = MODEL_DIRS[model_name] / f"prediction_{dataset}.csv"

    train_df = pd.read_csv(train_path)
    real_df = pd.read_csv(test_path)
    pred_df = pd.read_csv(pred_path)

    with open(info_path, "r") as f:
        info = json.load(f)

    return train_df, real_df, pred_df, info, pred_path


In [9]:

real_df, pred_df, info, pred_path = load_real_and_prediction("adult", "tabsyn")
print(pred_path)
print(real_df.shape, pred_df.shape)
print(compute_alpha_beta(real_df, pred_df, info))
print(compute_c2st(real_df, pred_df, info))


/Users/romandegtarev/generative_tabels/tabsyn/prediction_adult.csv
(16281, 15) (16281, 15)
{'alpha_precision': 0.9338083246319842, 'beta_recall': 0.3475953565505805}
0.784986115472679


## прогон по всем моделям и датасетам

In [19]:

rows = []

for dataset in DATASETS:
    for model_name in MODEL_DIRS:
        print(f"Processing dataset={dataset}, model={model_name}")

        train_df, real_df, pred_df, info, pred_path = load_train_real_and_prediction(dataset, model_name)
        ab = compute_alpha_beta(train_df, real_df, pred_df, info)
        c2st = compute_c2st(real_df, pred_df, info)

        rows.append({
            "dataset": dataset,
            "model": model_name,
            "alpha_precision": ab["alpha_precision"],
            "beta_recall": ab["beta_recall"],
            "c2st_logistic_detection": c2st,
        })


results_df = pd.DataFrame(rows)
results_df


Processing dataset=adult, model=tabsyn
Processing dataset=adult, model=tabdiff
Processing dataset=adult, model=tabcsdi
Processing dataset=adult, model=catboost
Processing dataset=default, model=tabsyn
Processing dataset=default, model=tabdiff
Processing dataset=default, model=tabcsdi
Processing dataset=default, model=catboost
Processing dataset=shoppers, model=tabsyn
Processing dataset=shoppers, model=tabdiff
Processing dataset=shoppers, model=tabcsdi
Processing dataset=shoppers, model=catboost
Processing dataset=magic, model=tabsyn
Processing dataset=magic, model=tabdiff
Processing dataset=magic, model=tabcsdi
Processing dataset=magic, model=catboost
Processing dataset=beijing, model=tabsyn
Processing dataset=beijing, model=tabdiff
Processing dataset=beijing, model=tabcsdi
Processing dataset=beijing, model=catboost


,dataset,model,alpha_precision,beta_recall,c2st_logistic_detection
0,adult,tabsyn,0.921938,0.306750,0.829286
1,adult,tabdiff,0.976913,0.222591,0.989144
2,adult,tabcsdi,0.983107,0.365768,0.865360
3,adult,catboost,0.611895,0.124190,0.300511
4,default,tabsyn,0.986742,0.425422,0.977515
5,default,tabdiff,0.892819,0.319867,0.928965
6,default,tabcsdi,0.965085,0.332289,0.826993
7,default,catboost,0.864289,0.343311,0.647127
8,shoppers,tabsyn,0.881427,0.445958,0.836730
9,shoppers,tabdiff,0.936794,0.424601,0.866926


## Сохранение результатов

In [20]:

results_df.to_csv(OUTPUT_CSV, index=False)
print("Saved to:", OUTPUT_CSV)


Saved to: /Users/romandegtarev/generative_tabels/den_evaluation_results/c2st_alpha_beta_results.csv
